## import necessary libraries

In [39]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel , RFE
from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier

## import dataset

In [40]:
breast_cancer = load_breast_cancer()
breast_cancer_df = pd.DataFrame(breast_cancer.data,columns = breast_cancer.feature_names)
breast_cancer_df["target"] = breast_cancer.target

In [41]:
X = breast_cancer_df.drop("target",axis = 1)
y = breast_cancer_df["target"]

## Data Understanding

In [42]:
breast_cancer_df.shape

(569, 31)

In [43]:
breast_cancer_df.isna().sum()

mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
target                     0
dtype: int64

In [44]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=123,shuffle=True,stratify=y)

## Model Building,Testing and Evaluation

In [45]:
def runRandomForestClassifier(X_train,X_test,y_train,y_test):
    rf_model = RandomForestClassifier(max_depth = 7,random_state = 123)
    rf_model.fit(X_train,y_train)
    
    y_pred = rf_model.predict(X_test)
    
    print("Accuracy Score:",round(accuracy_score(y_test,y_pred),4))
    print("Precision Score:",round(precision_score(y_test,y_pred),4))
    print("Recall Score",round(recall_score(y_test,y_pred),4))
    print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))

## Feature Selection using SelectFromModel

In [46]:
select_from_model = SelectFromModel(estimator = RandomForestClassifier(),max_features=12,)
select_from_model.fit(X_train,y_train)

,estimator,RandomForestClassifier()
,threshold,None
,prefit,False
,norm_order,1
,max_features,12
,importance_getter,'auto'
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1


In [47]:
select_from_model.get_support()

array([ True, False,  True,  True, False, False,  True,  True, False,
       False, False, False, False,  True, False, False, False, False,
       False, False,  True, False,  True,  True, False, False, False,
        True, False, False])

In [48]:
breast_cancer.feature_names[select_from_model.get_support()]

array(['mean radius', 'mean perimeter', 'mean area', 'mean concavity',
       'mean concave points', 'area error', 'worst radius',
       'worst perimeter', 'worst area', 'worst concave points'],
      dtype='<U23')

In [49]:
len(breast_cancer.feature_names[select_from_model.get_support()])

10

In [50]:
X_train_select_from_model = select_from_model.transform(X_train)
X_test_select_from_model = select_from_model.transform(X_test)

In [51]:
X_train_select_from_model.shape,X_test_select_from_model.shape

((455, 10), (114, 10))

In [52]:
%%time
runRandomForestClassifier(X_train,X_test,y_train,y_test)

Accuracy Score: 0.9649
Precision Score: 0.9722
Recall Score 0.9722
Confusion Matrix:
 [[40  2]
 [ 2 70]]
CPU times: total: 188 ms
Wall time: 197 ms


In [53]:
%%time
runRandomForestClassifier(X_train_select_from_model,X_test_select_from_model,y_train,y_test)

Accuracy Score: 0.9474
Precision Score: 0.9714
Recall Score 0.9444
Confusion Matrix:
 [[40  2]
 [ 4 68]]
CPU times: total: 156 ms
Wall time: 155 ms


## Using Recursive Feature Elimination

In [57]:
rfe_model = RFE(estimator = RandomForestClassifier(),n_features_to_select=12)
rfe_model.fit(X_train,y_train)

,estimator,RandomForestClassifier()
,n_features_to_select,12
,step,1
,verbose,0
,importance_getter,'auto'
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [59]:
rfe_model.get_support()

array([ True,  True,  True,  True, False, False,  True,  True, False,
       False, False, False, False,  True, False, False, False, False,
       False, False,  True, False,  True,  True, False, False,  True,
        True, False, False])

In [60]:
breast_cancer.feature_names[rfe_model.get_support()]

array(['mean radius', 'mean perimeter', 'mean area', 'mean concavity',
       'mean concave points', 'area error', 'worst radius',
       'worst perimeter', 'worst area', 'worst concave points'],
      dtype='<U23')

In [77]:
len(breast_cancer.feature_names[rfe_model.get_support()])

12

In [78]:
X_train_rfe_model = rfe_model.transform(X_train)
X_test_rfe_model = rfe_model.transform(X_test)

In [79]:
X_train_rfe_model.shape,X_test_rfe_model.shape

((455, 12), (114, 12))

In [80]:
%%time
runRandomForestClassifier(X_train,X_test,y_train,y_test)

Accuracy Score: 0.9649
Precision Score: 0.9722
Recall Score 0.9722
Confusion Matrix:
 [[40  2]
 [ 2 70]]
CPU times: total: 219 ms
Wall time: 204 ms


In [65]:
%%time
runRandomForestClassifier(X_train_rfe_model,X_test_rfe_model,y_train,y_test)

Accuracy Score: 0.9474
Precision Score: 0.9714
Recall Score 0.9444
Confusion Matrix:
 [[40  2]
 [ 4 68]]
CPU times: total: 141 ms
Wall time: 157 ms


## Take important features from Gradient Boosting and train it with Random Forest

In [66]:
rfe_modelGB = RFE(estimator = GradientBoostingClassifier(),n_features_to_select=12)
rfe_modelGB.fit(X_train,y_train)

,estimator,GradientBoostingClassifier()
,n_features_to_select,12
,step,1
,verbose,0
,importance_getter,'auto'
,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2


In [67]:
rfe_modelGB.get_support()

array([False,  True, False, False, False, False,  True,  True, False,
       False,  True,  True, False,  True, False, False, False, False,
       False, False, False,  True,  True,  True,  True, False,  True,
        True, False, False])

In [68]:
breast_cancer.feature_names[rfe_modelGB.get_support()]

array(['mean texture', 'mean concavity', 'mean concave points',
       'radius error', 'texture error', 'area error', 'worst texture',
       'worst perimeter', 'worst area', 'worst smoothness',
       'worst concavity', 'worst concave points'], dtype='<U23')

In [69]:
len(breast_cancer.feature_names[rfe_modelGB.get_support()])

12

In [70]:
X_train_rfe_modelGB = rfe_modelGB.transform(X_train)
X_test_rfe_modelGB = rfe_modelGB.transform(X_test)

In [71]:
X_train_rfe_modelGB.shape,X_test_rfe_modelGB.shape

((455, 12), (114, 12))

In [75]:
%%time
runRandomForestClassifier(X_train,X_test,y_train,y_test)

Accuracy Score: 0.9649
Precision Score: 0.9722
Recall Score 0.9722
Confusion Matrix:
 [[40  2]
 [ 2 70]]
CPU times: total: 203 ms
Wall time: 211 ms


In [76]:
%%time
runRandomForestClassifier(X_train_rfe_modelGB,X_test_rfe_modelGB,y_train,y_test)

Accuracy Score: 0.9737
Precision Score: 0.9859
Recall Score 0.9722
Confusion Matrix:
 [[41  1]
 [ 2 70]]
CPU times: total: 172 ms
Wall time: 166 ms


## choosing optimal number of features to train my model

In [ ]:
for i in range(1,31):
    rfe_modelGB = RFE(estimator = GradientBoostingClassifier(),n_features_to_select = i)
    rfe_modelGB.fit(X_train,y_train)
    rfe_modelGB.get_support()
    breast_cancer.feature_names[rfe_modelGB.get_support()]
    X_train_rfe_modelGB = rfe_modelGB.transform(X_train)
    X_test_rfe_modelGB = rfe_modelGB.transform(X_test)
    print("Selected Features:",i)
    runRandomForestClassifier(X_train_rfe_modelGB,X_test_rfe_modelGB,y_train,y_test)
    print("*************************")

Selected Features: 1
Accuracy Score: 0.886
Precision Score: 0.9041
Recall Score 0.9167
Confusion Matrix:
 [[35  7]
 [ 6 66]]
*************************
Selected Features: 2
Accuracy Score: 0.9123
Precision Score: 0.9189
Recall Score 0.9444
Confusion Matrix:
 [[36  6]
 [ 4 68]]
*************************
Selected Features: 3
Accuracy Score: 0.9561
Precision Score: 0.9718
Recall Score 0.9583
Confusion Matrix:
 [[40  2]
 [ 3 69]]
*************************
Selected Features: 4
Accuracy Score: 0.9474
Precision Score: 0.9714
Recall Score 0.9444
Confusion Matrix:
 [[40  2]
 [ 4 68]]
*************************
